In [1]:
print("hello")

hello


In [5]:
import requests
import time
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from pathlib import Path
from collections import deque


# ============================================================
# CONFIGURATION
# ============================================================

BASE_URL = "https://www.odoo.com"

OUTPUT_DIR = Path("odoo19_community_docs")

REQUEST_DELAY = 1.0
TIMEOUT = 30

# IMPORTANT:
# Keep this at 20 for testing.
# Later we can remove the limit.
MAX_PAGES = 20


# ============================================================
# COMMUNITY DOCUMENTATION ROOTS
# ============================================================

COMMUNITY_ROOTS = [

    # Essentials
    "/documentation/19.0/applications/essentials/",

    # Finance
    "/documentation/19.0/applications/finance/accounting/",
    "/documentation/19.0/applications/finance/expenses/",

    # Sales
    "/documentation/19.0/applications/sales/crm/",
    "/documentation/19.0/applications/sales/sales/",
    "/documentation/19.0/applications/sales/point_of_sale/",

    # Websites
    "/documentation/19.0/applications/websites/website/",
    "/documentation/19.0/applications/websites/ecommerce/",
    "/documentation/19.0/applications/websites/elearning/",
    "/documentation/19.0/applications/websites/forum/",
    "/documentation/19.0/applications/websites/blog/",
    "/documentation/19.0/applications/websites/live_chat/",

    # Supply Chain
    "/documentation/19.0/applications/inventory_and_mrp/inventory/",
    "/documentation/19.0/applications/inventory_and_mrp/purchase/",
    "/documentation/19.0/applications/inventory_and_mrp/manufacturing/",

    # HR
    "/documentation/19.0/applications/hr/attendances/",
    "/documentation/19.0/applications/hr/employees/",
    "/documentation/19.0/applications/hr/time_off/",
    "/documentation/19.0/applications/hr/recruitment/",

    # Services
    "/documentation/19.0/applications/services/project/",

    # Productivity
    "/documentation/19.0/applications/productivity/calendar/",
    "/documentation/19.0/applications/productivity/discuss/",

    # General
    "/documentation/19.0/applications/general/users.html",
    "/documentation/19.0/applications/general/companies.html",
]


# ============================================================
# ENTERPRISE EXCLUSIONS
# ============================================================

ENTERPRISE_ROOTS = [

    "/documentation/19.0/applications/sales/subscriptions/",
    "/documentation/19.0/applications/sales/rental/",

    "/documentation/19.0/applications/services/field_service/",
    "/documentation/19.0/applications/services/helpdesk/",
    "/documentation/19.0/applications/services/planning/",
    "/documentation/19.0/applications/services/timesheets/",

    "/documentation/19.0/applications/inventory_and_mrp/quality/",
    "/documentation/19.0/applications/inventory_and_mrp/maintenance/",

    "/documentation/19.0/applications/marketing/",

    "/documentation/19.0/applications/studio/",

    "/documentation/19.0/applications/productivity/documents/",
    "/documentation/19.0/applications/productivity/sign/",
    "/documentation/19.0/applications/productivity/spreadsheet/",
    "/documentation/19.0/applications/productivity/dashboards/",
    "/documentation/19.0/applications/productivity/knowledge/",
    "/documentation/19.0/applications/productivity/appointments/",
    "/documentation/19.0/applications/productivity/data_cleaning/",
    "/documentation/19.0/applications/productivity/iot/",
    "/documentation/19.0/applications/productivity/phone/",
    "/documentation/19.0/applications/productivity/voip/",
    "/documentation/19.0/applications/productivity/ai/",
]


# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Operose-Odoo19-Community-Documentation-Crawler/1.0"
    )
})


# ============================================================
# URL NORMALIZATION
# ============================================================

def normalize_url(url):

    parsed = urlparse(url)

    if parsed.netloc not in [
        "www.odoo.com",
        "odoo.com"
    ]:
        return None

    path = parsed.path

    # Remove fragment
    # Example:
    # page.html#section
    # becomes:
    # page.html

    if not path:
        return None

    # Remove trailing slash
    if path != "/" and path.endswith("/"):
        path = path[:-1]

    return f"https://www.odoo.com{path}"


# ============================================================
# CHECK COMMUNITY URL
# ============================================================

def is_community_url(url):

    if not url:
        return False

    parsed = urlparse(url)

    if parsed.netloc != "www.odoo.com":
        return False

    path = parsed.path

    # Must be Odoo 19 documentation
    if not path.startswith(
        "/documentation/19.0/"
    ):
        return False

    # --------------------------------------------------------
    # Enterprise exclusion
    # --------------------------------------------------------

    for enterprise_root in ENTERPRISE_ROOTS:

        if path.startswith(
            enterprise_root
        ):
            return False

    # --------------------------------------------------------
    # Community inclusion
    # --------------------------------------------------------

    for community_root in COMMUNITY_ROOTS:

        # Normal documentation directory
        if community_root.endswith("/"):

            if path.startswith(
                community_root
            ):
                return True

        # Individual HTML page
        else:

            if path == community_root:
                return True

    return False


# ============================================================
# FETCH PAGE
# ============================================================

def fetch_page(url):

    print()
    print(f"Downloading: {url}")

    response = session.get(
        url,
        timeout=TIMEOUT
    )

    response.raise_for_status()

    return BeautifulSoup(
        response.text,
        "html.parser"
    )


# ============================================================
# EXTRACT ARTICLE CONTENT
# ============================================================

def extract_article(soup):

    # --------------------------------------------------------
    # First try common article containers
    # --------------------------------------------------------

    article = soup.find("article")

    if article:
        return article


    # --------------------------------------------------------
    # Try role="main"
    # --------------------------------------------------------

    article = soup.find(
        attrs={"role": "main"}
    )

    if article:
        return article


    # --------------------------------------------------------
    # Fall back to <main>
    # --------------------------------------------------------

    return soup.find("main")


# ============================================================
# CLEAN ARTICLE
# ============================================================

def clean_article(article):

    if not article:
        return None

    # Remove things that aren't documentation content.
    #
    # We don't want scripts/styles/etc. in our RAG files.

    for tag in article.find_all(
        [
            "script",
            "style",
            "noscript"
        ]
    ):
        tag.decompose()


    # --------------------------------------------------------
    # Remove navigation elements
    # --------------------------------------------------------

    for selector in [
        "nav",
        ".breadcrumb",
        ".breadcrumbs",
        ".wy-breadcrumbs",
        ".headerlink",
        ".o_doc_breadcrumb",
    ]:

        for element in article.select(
            selector
        ):
            element.decompose()


    return article


# ============================================================
# SAVE PAGE
# ============================================================

def save_page(url, soup):

    article = extract_article(
        soup
    )

    if not article:

        print(
            f"WARNING: Article content not found: {url}"
        )

        return False


    article = clean_article(
        article
    )

    parsed = urlparse(url)

    path = parsed.path.lstrip("/")

    output_path = (
        OUTPUT_DIR / path
    )


    # --------------------------------------------------------
    # Create filename
    # --------------------------------------------------------

    if output_path.suffix != ".html":

        output_path = (
            output_path / "index.html"
        )


    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )


    # --------------------------------------------------------
    # Save HTML
    # --------------------------------------------------------

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            article.prettify()
        )


    print(
        f"Saved: {output_path}"
    )

    return True


# ============================================================
# EXTRACT LINKS
# ============================================================

def extract_links(
    current_url,
    soup
):

    links = set()

    # --------------------------------------------------------
    # IMPORTANT:
    # Find links from the documentation content,
    # not from the whole page.
    # --------------------------------------------------------

    article = extract_article(
        soup
    )

    if not article:
        return links


    # --------------------------------------------------------
    # Remove obvious navigation elements
    # before extracting links.
    # --------------------------------------------------------

    for element in article.find_all(
        [
            "nav",
            "footer"
        ]
    ):

        element.decompose()


    # --------------------------------------------------------
    # Extract links
    # --------------------------------------------------------

    for tag in article.find_all(
        "a",
        href=True
    ):

        href = tag["href"].strip()

        # Ignore empty links
        if not href:
            continue

        # Ignore javascript
        if href.startswith(
            "javascript:"
        ):
            continue

        absolute_url = urljoin(
            current_url,
            href
        )

        normalized = normalize_url(
            absolute_url
        )

        if not normalized:
            continue

        if is_community_url(
            normalized
        ):

            links.add(
                normalized
            )


    return links


# ============================================================
# CRAWLER
# ============================================================

def crawl():

    queue = deque()

    # --------------------------------------------------------
    # NEW:
    # Track URLs already placed in queue.
    # --------------------------------------------------------

    queued = set()

    visited = set()

    failed = set()


    # --------------------------------------------------------
    # Add starting Community roots
    # --------------------------------------------------------

    for root in COMMUNITY_ROOTS:

        if root.endswith(".html"):

            url = (
                BASE_URL + root
            )

        else:

            url = (
                BASE_URL
                + root.rstrip("/")
                + ".html"
            )


        normalized = normalize_url(
            url
        )

        if normalized:

            queue.append(
                normalized
            )

            queued.add(
                normalized
            )


    # --------------------------------------------------------
    # Crawl
    # --------------------------------------------------------

    while queue and len(visited) < MAX_PAGES:

        current_url = queue.popleft()


        if current_url in visited:
            continue


        print()
        print(
            "=" * 70
        )

        print(
            f"Progress: "
            f"{len(visited) + 1}/{MAX_PAGES}"
        )


        try:

            # ------------------------------------------------
            # Download
            # ------------------------------------------------

            soup = fetch_page(
                current_url
            )


            # ------------------------------------------------
            # Save
            # ------------------------------------------------

            save_page(
                current_url,
                soup
            )


            # ------------------------------------------------
            # Mark visited
            # ------------------------------------------------

            visited.add(
                current_url
            )


            # ------------------------------------------------
            # Discover links
            # ------------------------------------------------

            new_links = extract_links(
                current_url,
                soup
            )


            # ------------------------------------------------
            # Add only genuinely new links
            # ------------------------------------------------

            added = 0

            for link in new_links:

                if (
                    link not in visited
                    and link not in queued
                ):

                    queue.append(
                        link
                    )

                    queued.add(
                        link
                    )

                    added += 1


            print(
                f"New links added: {added}"
            )

            print(
                f"Queue size: {len(queue)}"
            )

            print(
                f"Visited: {len(visited)}"
            )


            # ------------------------------------------------
            # Respect server
            # ------------------------------------------------

            time.sleep(
                REQUEST_DELAY
            )


        except Exception as e:

            print(
                f"ERROR: {current_url}"
            )

            print(
                repr(e)
            )

            failed.add(
                current_url
            )

            visited.add(
                current_url
            )


    # ========================================================
    # SUMMARY
    # ========================================================

    print()
    print("=" * 70)
    print("CRAWLING COMPLETE")
    print("=" * 70)

    print(
        f"Pages visited : {len(visited)}"
    )

    print(
        f"Pages queued  : {len(queue)}"
    )

    print(
        f"Pages failed  : {len(failed)}"
    )

    if failed:

        print()
        print("Failed pages:")

        for url in failed:

            print(
                f" - {url}"
            )


# ============================================================
# RUN
# ============================================================

crawl()


Progress: 1/20

Downloading: https://www.odoo.com/documentation/19.0/applications/essentials.html
Saved: odoo19_community_docs/documentation/19.0/applications/essentials.html
New links added: 12
Queue size: 35
Visited: 1

Progress: 2/20

Downloading: https://www.odoo.com/documentation/19.0/applications/finance/accounting.html
Saved: odoo19_community_docs/documentation/19.0/applications/finance/accounting.html
New links added: 66
Queue size: 100
Visited: 2

Progress: 3/20

Downloading: https://www.odoo.com/documentation/19.0/applications/finance/expenses.html
Saved: odoo19_community_docs/documentation/19.0/applications/finance/expenses.html
New links added: 9
Queue size: 108
Visited: 3

Progress: 4/20

Downloading: https://www.odoo.com/documentation/19.0/applications/sales/crm.html
Saved: odoo19_community_docs/documentation/19.0/applications/sales/crm.html
New links added: 22
Queue size: 129
Visited: 4

Progress: 5/20

Downloading: https://www.odoo.com/documentation/19.0/applications/s

In [6]:
url = "https://www.odoo.com/documentation/19.0/applications/general/users.html"

soup = fetch_page(url)

save_page(url, soup)


Downloading: https://www.odoo.com/documentation/19.0/applications/general/users.html
Saved: odoo19_community_docs/documentation/19.0/applications/general/users.html


True